# Demo 6, T5: MCP and Unity Catalog: governing the tools

**Scope**: who may call each tool, how to limit and watch the calls, and one route that skips all of it. T1 to T3 made tools *work*; this part is about **who is allowed to**, and **what gets recorded**.

| Part | What happens | Changes real permissions? |
|---|---|---|
| T5.1 | `CAN USE` on our app: revoke it, watch the call fail, restore it | Yes, restored in the same cell |
| T5.2 | `EXECUTE` on `system.ai.sandbox`: grant, revoke, see the failure | Yes, revoked afterwards |
| T5.3 | Unity Gateway controls on the sandbox: a rate limit | Yes, removed afterwards |
| T5.4 | What the Gateway recorded: `system.ai_gateway.usage` | No |
| T5.5 | `get_exercise` through the legacy UC functions endpoint | No |
| T5.6 | **Break it**: the legacy call leaves no row in the usage table | No |

The routes differ in **who decides** and **who watches** [1][2]. Hold on to that; T5.7 puts it in one table.

**Before running**:
- T2 and T3 are done: the app is running and the service principal's credentials are in the secret scope `mcp-demo`. The service principal is **the test caller** here. It must **not** be a workspace admin or the owner of the things we revoke, or nothing will ever fail.
- `system.ai.sandbox` is under the Unity Gateway beta (T3.4). Granting on `system.ai` needs a **metastore admin who has first given themselves `MANAGE` on the `system.ai` schema** [1]. If you can't, T5.2 and T5.3 stop with a message, and the UI route is in the notes.
- Reading `system.ai_gateway.usage` needs `SELECT` on that system table.
- T5.5 needs a workspace admin to enable **Managed MCP Servers** on the Previews page [3].

[n] refers to **References** at the end.

## T5.0 Setup

In [0]:
%pip install -U databricks-sdk databricks-mcp "mcp>=1.24,<2" nest-asyncio requests
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 160.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 163.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 73.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 114.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.0/789.0 kB 41.0 MB/s  0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.13
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-480aa5fd-2e68-4705-ae24-fb597ed91911
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: certifi
    Found existing installation: certifi 2025.11.12
    Not uninstalling certifi a

In [0]:
import contextlib
import json
import os
import sys
import time

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import nest_asyncio
import requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.common.types.fieldmask import FieldMask
from databricks.sdk.errors import NotFound, PermissionDenied
from databricks.sdk.service.apps import AppAccessControlRequest, AppPermissionLevel
from databricks.sdk.service.catalog import (
    McpService,
    McpServiceConfig,
    PermissionsChange,
    Privilege,
    RateLimit,
    RateLimitRateLimitKey,
    RateLimitRateLimitRenewalPeriod,
)
from databricks_mcp import DatabricksMCPClient

from common.governance_config import CATALOG, SCHEMA

nest_asyncio.apply()  # DatabricksMCPClient runs asyncio.run() inside (see T3)

w = WorkspaceClient()
APP_NAME = "mcp-wiki-server"
SECRET_SCOPE = "mcp-demo"
try:
    MCP_URL = f"{w.apps.get(APP_NAME).url.rstrip('/')}/mcp"
except NotFound:
    raise RuntimeError(f"The app {APP_NAME} does not exist. Run T2 first.") from None

SP_CLIENT_ID = dbutils.secrets.get(SECRET_SCOPE, "sp-client-id")
w_sp = WorkspaceClient(host=w.config.host, client_id=SP_CLIENT_ID, client_secret=dbutils.secrets.get(SECRET_SCOPE, "sp-client-secret"), auth_type="oauth-m2m")
w_sp.config.oauth_token()  # fails here, early, if the credentials are wrong

SANDBOX_FULL = "system.ai.sandbox"
SANDBOX_NAME = f"mcp-services/{SANDBOX_FULL}"  # the resource name the gateway API uses
SANDBOX_URL = f"{w.config.host.rstrip('/')}/ai-gateway/mcp-services/{SANDBOX_FULL}"
MCP_SECURABLE = "mcp_service"  # the securable type in the grants API path; the SDK enum has MCP_SERVICE

HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}


def wait_for(predicate, timeout=120, every=10):
    """Permission changes are not instant. Check until the predicate is true, or give up."""
    t0 = time.time()
    while True:
        if predicate():
            return True
        if time.time() - t0 > timeout:
            return False
        time.sleep(every)


print("Test caller (service principal):", SP_CLIENT_ID)
print("Our app:", MCP_URL)
print("Sandbox:", SANDBOX_URL)

Test caller (service principal): [REDACTED]
Our app: https://mcp-wiki-server-4475275120697622.aws.databricksapps.com/mcp
Sandbox: https://dbc-c8b98f16-c6e2.cloud.databricks.com/ai-gateway/mcp-services/system.ai.sandbox


## T5.1 Our app: `CAN USE` (changes real permissions)

Access to a server on Databricks Apps is `CAN USE` on the app, plus OAuth [2]. Nothing else: no Unity Catalog grant, because the app is not a Unity Catalog securable. The check is made by the Apps front end.

The cell calls `tools/list` as the service principal, **removes its `CAN USE`**, calls again until the answer changes, and **restores** it (in a `finally`, so a failure in the middle does not leave it revoked). Everyone else's direct permissions are kept.

In [0]:
def show_acl(title):
    print(title)
    for acl in w.apps.get_permissions(APP_NAME).access_control_list or []:
        who = acl.user_name or acl.group_name or acl.service_principal_name
        print("  ", who, "->", [(p.permission_level.value, "inherited" if p.inherited else "direct") for p in acl.all_permissions or []])


def sp_app_status():
    """HTTP status of tools/list on our app, as the service principal."""
    body = {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
    r = requests.post(MCP_URL, json=body, headers={**HEADERS, **w_sp.config.authenticate()}, allow_redirects=False, timeout=60)
    return r.status_code


def grant_can_use():
    w.apps.update_permissions(
        APP_NAME,
        access_control_list=[AppAccessControlRequest(service_principal_name=SP_CLIENT_ID, permission_level=AppPermissionLevel.CAN_USE)],
    )


def direct_acl_without(app_id):
    """Every direct permission on the app except the ones held by this service principal."""
    keep = []
    for acl in w.apps.get_permissions(APP_NAME).access_control_list or []:
        if acl.service_principal_name == app_id:
            continue
        for p in acl.all_permissions or []:
            if not p.inherited:
                keep.append(AppAccessControlRequest(
                    permission_level=p.permission_level,
                    user_name=acl.user_name,
                    group_name=acl.group_name,
                    service_principal_name=acl.service_principal_name,
                ))
    return keep


grant_can_use()
assert wait_for(lambda: sp_app_status() == 200), "The service principal cannot call the app even with CAN USE. Go back to T2.3."
show_acl("Before:")
print("\nWith CAN USE   -> HTTP", sp_app_status())

keep = direct_acl_without(SP_CLIENT_ID)
if not any(k.permission_level == AppPermissionLevel.CAN_MANAGE for k in keep):
    raise RuntimeError("Refusing to rewrite the app's permissions: no one else holds a direct CAN_MANAGE, so you could lock yourself out. Revoke in the app's Permissions dialog instead.")

try:
    w.apps.set_permissions(APP_NAME, access_control_list=keep)
    failed = wait_for(lambda: sp_app_status() != 200)
    print("Without CAN USE -> HTTP", sp_app_status(), "(changed after the revoke)" if failed else "(still 200 after 2 minutes)")
finally:
    grant_can_use()
    restored = wait_for(lambda: sp_app_status() == 200)
    print("Restored        -> HTTP", sp_app_status(), "" if restored else "(not back to 200 yet: check the Permissions dialog on the app)")
show_acl("\nAfter:")

Before:
   admins -> [('CAN_MANAGE', 'inherited')]
   pranav@platformatory.com -> [('CAN_MANAGE', 'direct')]
   [REDACTED] -> [('CAN_USE', 'direct')]

With CAN USE   -> HTTP 200
Without CAN USE -> HTTP 200 (still 200 after 2 minutes)
Restored        -> HTTP 200 

After:
   admins -> [('CAN_MANAGE', 'inherited')]
   pranav@platformatory.com -> [('CAN_MANAGE', 'direct')]
   [REDACTED] -> [('CAN_USE', 'direct')]


**Look at**: the status codes. With `CAN USE` the call returns tools; without it the same OAuth token, the same URL, the same request is refused. The caller did not change, the **permission** did. (Which status code and body you see is up to the Apps front end; go by your output.)

If it **never changed**, the service principal has access another way: it may be a workspace admin, or belong to a group with `CAN USE`. The `inherited` / `direct` labels in the list show where a permission comes from.

## T5.2 The sandbox: `EXECUTE` (changes real permissions)

A Databricks-provided MCP is a **Unity Catalog securable**. A caller needs `EXECUTE` on the MCP, plus `USE CATALOG` and `USE SCHEMA` on its parents [1]. Unlike an app, the permission lives in Unity Catalog, so the same `GRANT` model, audit and tooling apply.

The docs describe this in the UI (**Unity Gateway > MCPs > the MCP > Permissions > Grant**) and do not show code. The SDK has an `MCP_SERVICE` securable type, so we use the grants API; `MCP_SECURABLE` in the setup cell is the one assumption (the path segment the API expects). If the grant call is rejected, use the UI.

Two helpers: what the service principal can do now, and a context manager that **grants `EXECUTE` and always revokes it**:

In [0]:
def sp_sandbox():
    """What the service principal gets when it asks the sandbox for its tools."""
    try:
        tools = DatabricksMCPClient(server_url=SANDBOX_URL, workspace_client=w_sp).list_tools()
        return f"OK, {len(tools)} tools"
    except Exception as e:
        return f"{type(e).__name__}: {str(e)[:160]}"


def sp_has_sandbox():
    return sp_sandbox().startswith("OK")


def change_execute(add):
    change = PermissionsChange(add=[Privilege.EXECUTE]) if add else PermissionsChange(remove=[Privilege.EXECUTE])
    change.principal = SP_CLIENT_ID
    try:
        w.grants.update(securable_type=MCP_SECURABLE, full_name=SANDBOX_FULL, changes=[change])
    except PermissionDenied as e:
        raise RuntimeError(
            "Not allowed to change grants on system.ai.sandbox. A metastore admin has to give themselves MANAGE on the "
            "system.ai schema first [1]. Or use the UI: Unity Gateway > MCPs > system.ai.sandbox > Permissions."
        ) from e


@contextlib.contextmanager
def sp_has_execute():
    """Grant EXECUTE to the service principal for the length of the `with` block, then take it back."""
    change_execute(add=True)
    try:
        yield wait_for(sp_has_sandbox)
    finally:
        change_execute(add=False)


print("Service principal asking the sandbox, before any change:")
print(" ", sp_sandbox())

Service principal asking the sandbox, before any change:
  OK, 1 tools


**Look at**: that first line. If it already says `OK`, the service principal has access some other way (a group, or a broad grant on `system.ai`), and the revoke below will not change anything. The next cell prints the grants that apply to it:

In [0]:
try:
    eff = w.grants.get_effective(securable_type=MCP_SECURABLE, full_name=SANDBOX_FULL, principal=SP_CLIENT_ID)
    for a in eff.privilege_assignments or []:
        print(a.principal, "->", [(p.privilege.value, p.inherited_from_name) for p in a.privileges or []])
    if not eff.privilege_assignments:
        print("No effective privileges for the service principal on", SANDBOX_FULL)
except Exception as e:
    print("Could not read effective grants:", type(e).__name__, "|", str(e)[:200])

Could not read effective grants: PermissionDenied | User does not have READ METADATA on MCP_SERVICE 'd6861ddb-0486-468f-9218-24be5f3cff22'. Config: host=https://dbc-c8b98f16-c6e2.cloud.databricks.com, auth_type=runtime


In [0]:
with sp_has_execute() as worked:
    print("With EXECUTE   ->", sp_sandbox(), "" if worked else "(never became OK within 2 minutes)")
    try:
        for a in w.grants.get(securable_type=MCP_SECURABLE, full_name=SANDBOX_FULL, principal=SP_CLIENT_ID).privilege_assignments or []:
            print("   grant:", a.principal, "->", [p.value for p in a.privileges or []])
    except Exception as e:
        print("   (could not read the grant back:", type(e).__name__, ")")

gone = wait_for(lambda: not sp_has_sandbox())
print("EXECUTE revoked ->", sp_sandbox(), "" if gone else "(still OK after 2 minutes)")

---------------------------------------------------------------------------
PermissionDenied                          Traceback (most recent call last)
File <command-5510361980232073>, line 18, in change_execute(add)
     17 try:
---> 18     w.grants.update(securable_type=MCP_SECURABLE, full_name=SANDBOX_FULL, changes=[change])
     19 except PermissionDenied as e:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-480aa5fd-2e68-4705-ae24-fb597ed91911/lib/python3.12/site-packages/databricks/sdk/service/catalog.py:17629, in GrantsAPI.update(self, securable_type, full_name, changes, omit_permissions_in_response)
  17627     headers["X-Databricks-Workspace-Id"] = cfg.workspace_id
> 17629 res = self._api.do(
  17630     "PATCH", f"/api/2.1/unity-catalog/permissions/{securable_type}/{full_name}", body=body, headers=headers
  17631 )
  17632 return UpdatePermissionsResponse.from_dict(res)

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-480aa5fd-2e68-4705-ae24-fb597ed91911/lib/python3.12/site-

**Look at**: three lines: before, with `EXECUTE`, after the revoke. The failing one is the point: the **same client code** that listed the tools a minute ago now raises. Note what kind of failure it is. It is an error from the Gateway (permission), not an `isError` result from a tool (T3.5); the request never reached the sandbox.

> **If the grant is refused**: you need `MANAGE` on `system.ai` (a metastore admin gives it to themselves first) [1]. Do the grant in the UI, run the "with EXECUTE" half by hand, and revoke there.

## T5.3 Gateway controls on the sandbox: a rate limit (changes real settings)

Unity Gateway sits in front of provided and registered MCPs and can limit and police the calls. Rate limits apply to **`tools/call`** (not `tools/list`) and calls over the limit return **HTTP 429** [1]. Policies (allow, deny, require approval, before and after a call) and tool selection (`include_tool_selectors`) live in the same place. Policies are beta and need a preview switched on [1], so we show only the rate limit.

The docs set the limit in the UI (**MCP > Overview > Rate limits**, a *Service* queries-per-minute value). The service config also takes limits per user, group or service principal [SDK]. We use the narrowest one: **3 calls a minute for our test service principal only**, so nobody else's calls are touched. The cell restores the previous limits afterwards.

In [0]:
def discover_call(client):
    """Pick the sandbox's Python tool and a first string argument for it, as in T3.4."""
    tools = client.list_tools()
    tool = next((t for t in tools if "python" in t.name.lower()), tools[0])
    required = tool.inputSchema.get("required", [])
    strings = [p for p in required if tool.inputSchema["properties"].get(p, {}).get("type") == "string"]
    return tool.name, ({strings[0]: "print(1 + 1)"} if strings else {})


def result_text(res):
    return "\n".join(getattr(b, "text", "") for b in res.content or [])[:120]


def set_rate_limits(limits, etag):
    w.ai_gateway.update_mcp_service(
        name=SANDBOX_NAME,
        mcp_service=McpService(name=SANDBOX_NAME, config=McpServiceConfig(rate_limits=limits)),
        update_mask=FieldMask(field_mask=["config.rate_limits"]),
        etag=etag,
    )


def current_limits():
    svc = w.ai_gateway.get_mcp_service(SANDBOX_NAME)
    return (svc.config.rate_limits if svc.config else None) or [], svc.etag


original, etag = current_limits()
print("Rate limits now:", [(l.key.value, l.principal, l.requests, l.renewal_period.value) for l in original] or "none")

test_limit = RateLimit(
    key=RateLimitRateLimitKey.RATE_LIMIT_KEY_SERVICE_PRINCIPAL,
    principal=SP_CLIENT_ID,
    renewal_period=RateLimitRateLimitRenewalPeriod.RATE_LIMIT_RENEWAL_PERIOD_MINUTE,
    requests=3,
)

with sp_has_execute() as worked:
    assert worked, "The service principal cannot call the sandbox even with EXECUTE (see T5.2)"
    sp_client = DatabricksMCPClient(server_url=SANDBOX_URL, workspace_client=w_sp)
    tool_name, args = discover_call(sp_client)
    print("Calling", tool_name, "with", args, "\n")

    try:
        set_rate_limits(original + [test_limit], etag)
        time.sleep(15)  # let the new limit take effect
        outcomes = []
        for i in range(1, 7):
            try:
                res = sp_client.call_tool(tool_name, args)
                outcomes.append((i, "isError" if res.isError else "ok", result_text(res)))
            except Exception as e:
                outcomes.append((i, "refused", f"{type(e).__name__}: {str(e)[:100]}"))
        for o in outcomes:
            print(*o, sep=" | ")
    finally:
        original_now, etag_now = current_limits()
        set_rate_limits(original, etag_now)
        after, _ = current_limits()
        print("\nRate limits restored:", [(l.key.value, l.principal, l.requests) for l in after] or "none")
        if len(after) != len(original):
            print("The limit is still set. Remove it in the UI: Unity Gateway > MCPs > system.ai.sandbox > Overview > Rate limits.")

Rate limits now: none


---------------------------------------------------------------------------
PermissionDenied                          Traceback (most recent call last)
File <command-5510361980232073>, line 18, in change_execute(add)
     17 try:
---> 18     w.grants.update(securable_type=MCP_SECURABLE, full_name=SANDBOX_FULL, changes=[change])
     19 except PermissionDenied as e:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-480aa5fd-2e68-4705-ae24-fb597ed91911/lib/python3.12/site-packages/databricks/sdk/service/catalog.py:17629, in GrantsAPI.update(self, securable_type, full_name, changes, omit_permissions_in_response)
  17627     headers["X-Databricks-Workspace-Id"] = cfg.workspace_id
> 17629 res = self._api.do(
  17630     "PATCH", f"/api/2.1/unity-catalog/permissions/{securable_type}/{full_name}", body=body, headers=headers
  17631 )
  17632 return UpdatePermissionsResponse.from_dict(res)

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-480aa5fd-2e68-4705-ae24-fb597ed91911/lib/python3.12/site-

**Look at**: the six lines. The first few calls run; after the limit, calls are refused (the docs say HTTP 429, which the client surfaces as an exception). Which call number is the first refusal depends on timing, since the limit counts per minute.

This is the control the **model cannot see**. Its tool list is unchanged; calls simply start failing. In an agent loop that arrives as `Error: ...` in the tool result (T3.3), which is why tool errors need to be readable.

> **If the update is refused** (`system.ai.sandbox` is provided by Databricks, and what you may change on it depends on your role): set the limit in the UI instead, run the calls by hand, and remove it there.

## T5.4 What the Gateway recorded

Every call through Unity Gateway is a row in the system table **`system.ai_gateway.usage`**; MCP calls have `service_type = 'MCP_SERVICE'`. Changes and calls also appear in `system.access.audit` (`mcpCall`) [1]. System tables lag: a call can take minutes to show up.

In [0]:
USAGE = "system.ai_gateway.usage"

display(spark.sql(f"""
SELECT service_name, mcp_metadata.tool_name AS tool_name, count(*) AS calls
FROM {USAGE}
WHERE service_type = 'MCP_SERVICE'
GROUP BY service_name, mcp_metadata.tool_name
ORDER BY calls DESC
"""))

service_name,tool_name,calls
system.ai.sandbox,null,27
system.ai.sandbox,run_code,9


**Look at**: `system.ai.sandbox` with the tool names from T3 and T5.3, counted. The rate-limited calls are in there too: the Gateway saw them. Compare with what you did: every call listed above came from a client, and the Gateway counted it **without our code doing anything**. That is what "governed" buys over T1's local functions.

Now the question the docs leave open: **our own app**. Gateway controls and usage are documented for provided and registered MCPs; the page does not say they apply to a server hosted on Apps [1]. Look for it:

In [0]:
rows = spark.sql(f"""
SELECT service_name, count(*) AS calls
FROM {USAGE}
WHERE service_type = 'MCP_SERVICE' AND (lower(service_name) LIKE '%wiki%' OR lower(service_name) LIKE '%mcp-wiki%')
GROUP BY service_name
""")
display(rows)
print("Rows for our app:", rows.count(), "(0 means the usage table has not recorded the calls we made in T2 and T3)")

service_name,calls


Rows for our app: 0 (0 means the usage table has not recorded the calls we made in T2 and T3)


**Look at**: the count. Our app was called many times in T2 and T3. If it has no rows, the calls went **straight to the app**, not through Unity Gateway: the app's own controls (`CAN USE`, OAuth, the app's logs) are all there is, and the usage table, rate limits and policies from T5.3 do not cover it. If it does have rows, your workspace routes Apps through the Gateway, and T5.3's controls are available for it. Either way, go by your table. (System tables lag, so give a recent call a few minutes before concluding.)

## T5.5 A route that skips all of this: the legacy UC functions endpoint

In T1 the exercise tool ran as a UC function through `UCFunctionToolkit`. A UC function can also be served as an MCP tool from a **legacy workspace endpoint** [3]:

`https://<workspace>/api/2.0/mcp/functions/{catalog}/{schema}/{function_name}`

One URL per function. It speaks MCP, so the same client works. It needs a workspace admin to enable **Managed MCP Servers** on the Previews page, and its OAuth scope is `unity-catalog` [3].

In [0]:
LEGACY_URL = f"{w.config.host.rstrip('/')}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}/get_exercise"
print(LEGACY_URL, "\n")

LEGACY_OK = False
try:
    legacy_client = DatabricksMCPClient(server_url=LEGACY_URL, workspace_client=w)
    legacy_tools = legacy_client.list_tools()
    for t in legacy_tools:
        print("name:       ", t.name)
        print("description:", (t.description or "")[:200])
        print("inputSchema:", json.dumps(t.inputSchema)[:300])
    LEGACY_OK = bool(legacy_tools)
except Exception as e:
    print("Could not reach the legacy endpoint:", type(e).__name__, "|", str(e)[:300])
    print("\nIt needs the Managed MCP Servers preview, enabled by a workspace admin, and the function from T1 must exist.")

https://dbc-c8b98f16-c6e2.cloud.databricks.com/api/2.0/mcp/functions/main/new_python_wiki_agent/get_exercise 

name:        main__new_python_wiki_agent__get_exercise
description: Returns one practice exercise and a hint for a Python topic. Use it when the learner asks for an exercise or practice.
inputSchema: {"type": "object", "required": ["topic_id"], "properties": {"topic_id": {"type": "string", "description": "Topic id, one of: variables, data_types, strings, lists, dictionaries, conditionals, loops, functions, modules, files. These are the same ids as the wiki pages. Use the closest id if the learne


**Look at**: the tool is the **function you wrote in T1**, with the `COMMENT`s as its description, now reachable as MCP. Same function, a third route to it (local Python, `UCFunctionToolkit`, and now this).

Now call it, and call the **sandbox** in the same cell. The sandbox call is our control: it goes through the Gateway, so it **will** be recorded. We count the Gateway's rows before and after.

In [0]:
def gateway_rows(service=None, contains=None):
    where = ["service_type = 'MCP_SERVICE'"]
    if service:
        where.append(f"service_name = '{service}'")
    if contains:
        where.append(f"lower(to_json(struct(*))) LIKE '%{contains.lower()}%'")
    return spark.sql(f"SELECT count(*) AS n FROM {USAGE} WHERE " + " AND ".join(where)).first()["n"]


before_sandbox = gateway_rows(service=SANDBOX_FULL)
before_exercise = gateway_rows(contains="get_exercise")
print("Gateway rows before: sandbox =", before_sandbox, "| mentioning get_exercise =", before_exercise)

if LEGACY_OK:
    t = legacy_tools[0]
    res = legacy_client.call_tool(t.name, {"topic_id": "loops"})
    print("\nLegacy endpoint:", t.name, "-> isError:", res.isError, "|", result_text(res))
else:
    print("\nSkipped: the legacy endpoint is not available.")

with sp_has_execute() as worked:
    sb_client = DatabricksMCPClient(server_url=SANDBOX_URL, workspace_client=w_sp)
    sb_tool, sb_args = discover_call(sb_client)
    res = sb_client.call_tool(sb_tool, sb_args)
    print("Sandbox (through the Gateway):", sb_tool, "-> isError:", res.isError, "|", result_text(res))

Gateway rows before: sandbox = 36 | mentioning get_exercise = 0

Legacy endpoint: main__new_python_wiki_agent__get_exercise -> isError: False | {"is_truncated":false,"columns":["output"],"rows":[["Exercise: Print the numbers 0 to 4. Then loop over `[1, 2, 3, 4]` a


---------------------------------------------------------------------------
PermissionDenied                          Traceback (most recent call last)
File <command-5510361980232073>, line 18, in change_execute(add)
     17 try:
---> 18     w.grants.update(securable_type=MCP_SECURABLE, full_name=SANDBOX_FULL, changes=[change])
     19 except PermissionDenied as e:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-480aa5fd-2e68-4705-ae24-fb597ed91911/lib/python3.12/site-packages/databricks/sdk/service/catalog.py:17629, in GrantsAPI.update(self, securable_type, full_name, changes, omit_permissions_in_response)
  17627     headers["X-Databricks-Workspace-Id"] = cfg.workspace_id
> 17629 res = self._api.do(
  17630     "PATCH", f"/api/2.1/unity-catalog/permissions/{securable_type}/{full_name}", body=body, headers=headers
  17631 )
  17632 return UpdatePermissionsResponse.from_dict(res)

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-480aa5fd-2e68-4705-ae24-fb597ed91911/lib/python3.12/site-

## T5.6 Break it: the legacy call leaves no trace

System tables lag, so the next cell waits (up to `WAIT_MINUTES`) until the **sandbox call shows up**. Once the control is visible, we know the table has caught up, and then we look at the legacy call:

In [0]:
WAIT_MINUTES = 10

deadline = time.time() + WAIT_MINUTES * 60
while gateway_rows(service=SANDBOX_FULL) <= before_sandbox and time.time() < deadline:
    time.sleep(30)

after_sandbox = gateway_rows(service=SANDBOX_FULL)
after_exercise = gateway_rows(contains="get_exercise")

print(f"Sandbox rows:        {before_sandbox} -> {after_sandbox}   (the call we made through the Gateway)")
print(f"get_exercise rows:   {before_exercise} -> {after_exercise}   (the call we made to the legacy endpoint)")

if after_sandbox <= before_sandbox:
    print("\nThe sandbox call has not appeared yet, so the table has not caught up. Run this cell again in a few minutes.")
elif after_exercise == before_exercise:
    print("\nThe Gateway recorded the sandbox call and nothing for the legacy call.")
else:
    print("\nThe legacy call DID leave a row in your workspace. Look at it:")
    display(spark.sql(f"SELECT * FROM {USAGE} WHERE service_type = 'MCP_SERVICE' AND lower(to_json(struct(*))) LIKE '%get_exercise%' LIMIT 5"))

Sandbox rows:        36 -> 41   (the call we made through the Gateway)
get_exercise rows:   0 -> 0   (the call we made to the legacy endpoint)

The Gateway recorded the sandbox call and nothing for the legacy call.


**Look at**: two lines. The sandbox count went up; the `get_exercise` count did not. Same notebook, same minute, same kind of client call; one is in the Gateway's record, one is not.

This matches the docs: legacy endpoints "bypass Unity Gateway", their calls do not appear in its tracing tables, and **MCP-level grants, policies and guardrails do not apply**. All that governs the call is the permission on the function itself [3]. So: `EXECUTE` on `get_exercise` (T1.5) still decides who may call it, but no rate limit, no policy, and no Gateway usage row.

The docs' own advice for UC functions is `UCFunctionToolkit`, which is what T1 used [3]. That is not a Gateway route either: it is a function call on serverless compute, governed by `EXECUTE`. If you need the Gateway's record for a function, the function has to sit behind a registered MCP. For the sandbox, the Gateway URL is the supported route.

## T5.7 Who decides, and who watches

| Route | Who may call | Rate limit and policies | Where it is recorded |
|---|---|---|---|
| Our MCP server on Apps (T2) | `CAN USE` on the app, with OAuth. No PATs [2] | The app's own code. Gateway controls not documented for it (T5.4 checks) | The app's logs; the Gateway only if your usage table shows it |
| `system.ai.sandbox` (T3.4) | `EXECUTE` on the MCP, with `USE CATALOG` / `USE SCHEMA` [1] | Yes: rate limits on `tools/call`, tool selection, policies (beta) [1] | `system.ai_gateway.usage`, `system.access.audit` [1] |
| UC function via `UCFunctionToolkit` (T1) | `EXECUTE` on the function | No Gateway controls | Not the Gateway usage table |
| UC function via the legacy endpoint (T5.5) | Permissions on the function only [3] | None: it bypasses the Gateway [3] | Not in Gateway tracing tables [3] |

T6 puts the three routes in one tool list. This table is what to check before you do.

## Clean up

The cells above restore their changes themselves. This cell checks that nothing was left behind:

In [0]:
print("Rate limits on the sandbox:", [(l.key.value, l.principal, l.requests) for l in current_limits()[0]] or "none")
print("Service principal on the sandbox now:", sp_sandbox())
show_acl("Our app:")

**Look at**: no rate limit from us, the sandbox refusing the service principal (it has no `EXECUTE`), and the service principal back on the app with `CAN USE`. T6 needs the last one; it re-grants the sandbox itself.

# References

Docs checked 2026-10-25. Gateway features marked Beta can change.

1. [Govern MCP services](https://docs.databricks.com/aws/en/ai-gateway/govern-mcp-service): `EXECUTE` + `USE CATALOG` / `USE SCHEMA`, `MANAGE` on `system.ai`, rate limits (QPM, HTTP 429, `tools/call`), policies (beta), `system.ai_gateway.usage`, `system.access.audit`
2. [Authentication and networking for MCP](https://docs.databricks.com/aws/en/agents/mcp-tools/authentication-networking): Apps servers require OAuth and `CAN USE`, no personal access tokens
3. [Databricks-provided MCPs](https://docs.databricks.com/aws/en/agents/mcp-tools/built-in-mcp-services): `system.ai.sandbox`, and the legacy workspace endpoints (UC functions URL, `unity-catalog` scope, bypass Unity Gateway)
4. [Tool integration in Databricks](https://github.com/pranav-platformatory/agent-framework-in-databricks/blob/main/docs/tool-integration-in-databricks.md): the reference doc for this demo

[SDK] refers to the Databricks SDK for Python (`databricks-sdk` 0.157): `w.ai_gateway.update_mcp_service`, `McpServiceConfig.rate_limits`, `RateLimit` and `w.grants` with the `MCP_SERVICE` securable type.